# Meta-IDS 

**Dataset:** `wustl-ehms-2020_with_attacks_categories.csv`  


## 1. Import libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
DATA_PATH = "wustl-ehms-2020_with_attacks_categories.csv"
OUTPUT_DIR = Path("meta_ids_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Load and inspect dataset

In [12]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))
print("\nLabel distribution:")
display(df["Label"].value_counts())


Dataset shape: (16318, 45)


,Dir,Flgs,SrcAddr,DstAddr,Sport,Dport,SrcBytes,DstBytes,SrcLoad,DstLoad,...,Temp,SpO2,Pulse_Rate,SYS,DIA,Heart_rate,Resp_Rate,ST,Attack Category,Label
0,->,e,10.0.1.172,10.0.1.150,58059,1111,496,186,276914.0,92305.0,...,28.9,0,0,0,0,0,0,0.0,normal,0
1,->,e,10.0.1.172,10.0.1.150,58062,1111,496,186,230984.0,76995.0,...,28.9,0,0,0,0,78,17,0.4,normal,0
2,->,e,10.0.1.172,10.0.1.150,58065,1111,496,186,218470.0,72823.0,...,28.9,89,104,0,0,78,17,0.4,normal,0
3,->,e,10.0.1.172,10.0.1.150,58067,1111,496,186,203376.0,67792.0,...,28.9,89,104,0,0,79,17,0.4,normal,0
4,->,e,10.0.1.172,10.0.1.150,58069,1111,496,186,235723.0,78574.0,...,28.9,89,101,0,0,79,17,0.4,normal,0



Missing values:


Dir           0
Flgs          0
SrcAddr       0
DstAddr       0
Sport         0
Dport         0
SrcBytes      0
DstBytes      0
SrcLoad       0
DstLoad       0
SrcGap        0
DstGap        0
SIntPkt       0
DIntPkt       0
SIntPktAct    0
dtype: int64


Label distribution:


Label
0    14272
1     2046
Name: count, dtype: int64

## 3. Clean columns and separate features/target


In [3]:
drop_columns = [
    "Dir", "SrcAddr", "DstAddr", "Sport",
    "SrcMac", "DstMac", "Attack Category"
]
df = df.dropna(subset=["Label"]).copy()
df = df.drop(columns=[c for c in drop_columns if c in df.columns], errors="ignore")

X = df.drop(columns=["Label"])
y = df["Label"].astype(int)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
display(y.value_counts())


Feature shape: (16318, 37)
Target shape: (16318,)


Label
0    14272
1     2046
Name: count, dtype: int64

## 4. Spliting data

In [14]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train_raw.shape)
print("Test shape:", X_test_raw.shape)
print("\nTraining labels:")
display(y_train.value_counts())
print("Test labels:")
display(y_test.value_counts())


Train shape: (11422, 37)
Test shape: (4896, 37)

Training labels:


Label
0    9990
1    1432
Name: count, dtype: int64

Test labels:


Label
0    4282
1     614
Name: count, dtype: int64

## 5. Encode categorical columns
One-hot encoding is used for categorical columns. The encoding is created from training data, and test data is aligned to the same columns. Missing numeric values are filled with the training median.


In [ ]:
cat_cols = X_train_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X_train_raw.columns if c not in cat_cols]

for col in num_cols:
    median_value = X_train_raw[col].median()
    X_train_raw.loc[:, col] = X_train_raw[col].fillna(median_value)
    X_test_raw.loc[:, col] = X_test_raw[col].fillna(median_value)

for col in cat_cols:
    mode_values = X_train_raw[col].mode()
    fill_value = mode_values.iloc[0] if not mode_values.empty else "unknown"
    X_train_raw.loc[:, col] = X_train_raw[col].fillna(fill_value)
    X_test_raw.loc[:, col] = X_test_raw[col].fillna(fill_value)

X_train = pd.get_dummies(X_train_raw, columns=cat_cols)
X_test = pd.get_dummies(X_test_raw, columns=cat_cols)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

X_train = X_train.astype(float)
X_test = X_test.astype(float)

print("Train after encoding:", X_train.shape)
print("Test after encoding:", X_test.shape)


Train after encoding: (11422, 42)
Test after encoding: (4896, 42)


C:\Users\sg131\AppData\Local\Temp\ipykernel_30840\772806819.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


## 6. Scale features using training data only

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)


Scaled train shape: (11422, 42)
Scaled test shape: (4896, 42)


## 7. Apply SMOTE to training data only

In [7]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_scaled, y_train
)

print("Before SMOTE:")
display(y_train.value_counts())
print("After SMOTE:")
display(pd.Series(y_train_balanced).value_counts())
print("Test set remains unchanged:")
display(y_test.value_counts())


Before SMOTE:


Label
0    9990
1    1432
Name: count, dtype: int64

After SMOTE:


Label
1    9990
0    9990
Name: count, dtype: int64

Test set remains unchanged:


Label
0    4282
1     614
Name: count, dtype: int64

## 8. Select features using Random Forest RFE

In [8]:
number_of_features = min(15, X_train_balanced.shape[1])

rfe = RFE(
    estimator=RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    n_features_to_select=number_of_features
)

X_train_selected = rfe.fit_transform(X_train_balanced, y_train_balanced)
X_test_selected = rfe.transform(X_test_scaled)

selected_features = X_train.columns[rfe.support_].tolist()

print("Number of selected features:", len(selected_features))
print("Selected features:")
print(selected_features)
print("Train selected shape:", X_train_selected.shape)
print("Test selected shape:", X_test_selected.shape)


Number of selected features: 15
Selected features:
['SrcLoad', 'SIntPkt', 'DIntPkt', 'SrcJitter', 'DstJitter', 'Dur', 'Load', 'Rate', 'Packet_num', 'Temp', 'Pulse_Rate', 'Heart_rate', 'Resp_Rate', 'Flgs_ M        ', 'Flgs_ e        ']
Train selected shape: (19980, 15)
Test selected shape: (4896, 15)


## 9. Prepare six base learners
The paper names Decision Tree, Random Forest, AdaBoost, Multinomial Naive Bayes, NN, and MLP. Here, KNN is used as the nearest-neighbor model for **NN**. Multinomial Naive Bayes needs non-negative data, so it gets its own MinMax-scaled version.


In [9]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=100, random_state=RANDOM_STATE
    ),
    "KNN (NN)": KNeighborsClassifier(n_neighbors=5),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=300,
        random_state=RANDOM_STATE
    ),
    "Multinomial Naive Bayes": MultinomialNB()
}

# MNB requires non-negative values
mnb_scaler = MinMaxScaler()
X_train_mnb = mnb_scaler.fit_transform(X_train_selected)
X_test_mnb = mnb_scaler.transform(X_test_selected)


## 10. Train and evaluate base learners

In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    if name == "Multinomial Naive Bayes":
        Xtr, Xte = X_train_mnb, X_test_mnb
    else:
        Xtr, Xte = X_train_selected, X_test_selected

    print("Training:", name)
    model.fit(Xtr, y_train_balanced)
    predictions = model.predict(Xte)
    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False)
display(results_df.round(4))


Training: Decision Tree
Training: Random Forest
Training: AdaBoost
Training: KNN (NN)
Training: MLP
Training: Multinomial Naive Bayes


,Model,Accuracy,Precision,Recall,F1-score
1,Random Forest,0.9361,0.7385,0.7590,0.7486
0,Decision Tree,0.9281,0.6671,0.8518,0.7482
4,MLP,0.8971,0.5607,0.8274,0.6684
5,Multinomial Naive Bayes,0.9336,0.9801,0.4805,0.6448
3,KNN (NN),0.8746,0.5000,0.8469,0.6288
2,AdaBoost,0.7839,0.3506,0.8485,0.4962
